# 第7回　区間推定の理論
## ―― 信頼区間はどう作られ、独立が崩れるとどう嘘になるか

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

統計学Ⅰで「95%信頼区間」の **本当の意味**（真の値が95%の確率で入る、ではない）を学んだ。Ⅱでは、その区間が **どう作られるか** の数理に踏み込み、前回（第6回）の続きとして **独立が崩れると区間が嘘になる** ことを確かめる。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. 信頼区間の作り方

母平均 $\mu$ を、標本から推定したい。点推定は標本平均 $\bar{x}$ ひとつだが、それだけでは「どれくらい確かか」が分からない。そこで **区間** で示す。

$$\bar{x} \;\pm\; t_{0.975,\,n-1}\times \underbrace{\frac{s}{\sqrt{n}}}_{\text{標準誤差}}$$

- $s/\sqrt{n}$ … 第6回の **標準誤差**（標本平均のばらつき）
- $t_{0.975,\,n-1}$ … 母分散が未知なので、正規分布の代わりに **t分布** を使う（$n$ が小さいほど区間が広がる）

1つの標本から、実際に95%信頼区間を作ってみよう。

In [ ]:
rng = np.random.default_rng(7)
真の母平均, 母SD = 50, 10
n = 30
標本 = rng.normal(真の母平均, 母SD, n)

x̄ = 標本.mean()
s = 標本.std(ddof=1)              # 標本標準偏差
SE = s / np.sqrt(n)
t = stats.t.ppf(0.975, df=n - 1)  # t分布の95%点
下限, 上限 = x̄ - t * SE, x̄ + t * SE
print(f"標本平均 x̄ = {x̄:.2f}（点推定）")
print(f"標準誤差 SE = {SE:.2f}, t(0.975, 29) = {t:.3f}")
print(f"95%信頼区間 = [{下限:.2f}, {上限:.2f}]")
print(f"真の母平均 {真の母平均} は、この区間に入っている？ → {下限 <= 真の母平均 <= 上限}")

---
## 2. 「95%」の本当の意味 ―― カバレッジ

95%信頼区間の正しい意味は「真の値がこの区間に95%の確率で入る」では **ない**。正しくは：

> **同じ手続きで標本を取り直して区間を作る、を何度も繰り返すと、そのうち約95%の区間が真の値を含む。**

これを **カバレッジ（被覆率）** という。100本の区間を描いて、何本が真の母平均を含むか数えてみよう。

In [ ]:
def 信頼区間(標本, 信頼度=0.95):
    n = len(標本)
    SE = 標本.std(ddof=1) / np.sqrt(n)
    t = stats.t.ppf(0.5 + 信頼度 / 2, df=n - 1)
    m = 標本.mean()
    return m - t * SE, m + t * SE

plt.figure(figsize=(7, 6))
含んだ数 = 0
for i in range(100):
    標本 = rng.normal(真の母平均, 母SD, n)
    lo, hi = 信頼区間(標本)
    含む = lo <= 真の母平均 <= hi
    含んだ数 += 含む
    plt.plot([lo, hi], [i, i], color=("#3949ab" if 含む else "#e8503a"), lw=1.5)
plt.axvline(真の母平均, color="k", ls="--", label=f"真の母平均={真の母平均}")
plt.title(f"100本の95%信頼区間：{含んだ数}本が真値を含んだ（赤=外した）")
plt.xlabel("値"); plt.ylabel("何回目の標本か"); plt.legend()
plt.show()
print(f"100本中 {含んだ数} 本が真値を含んだ（理論は約95本）。これが『95%』の正体。")

約95本が真値を含み、約5本が外す。**「95%」は、真値についての確率ではなく、手続きの当たり率**だ（Ⅰの復習）。

区間の **幅** を決めるのは3つ：標準誤差 $s/\sqrt{n}$（データのばらつき）、信頼度（95%か99%か）、標本サイズ $n$。$n$ を増やすと幅は狭まる ―― **ただし、標本が独立なら**。

---
## 3. 独立が崩れると ―― 「95%」が嘘になる

第6回で見たとおり、標本が **相関する**（みんなが空気を読んで似た回答をする）と、本当のばらつきは $s/\sqrt{n}$ より大きい。なのに区間は $s/\sqrt{n}$ で作ってしまう ―― つまり **狭すぎる区間** になる。

狭すぎる区間は、真値をしょっちゅう外す。**カバレッジが95%を大きく割る**はずだ。独立な標本と相関した標本で、カバレッジを1万回ずつ測って比べよう。

In [ ]:
def カバレッジ(ρ, n=30, 試行=10000):
    含んだ = 0
    for _ in range(試行):
        共通 = rng.normal()
        個人 = rng.normal(size=n)
        標本 = 真の母平均 + 母SD * (np.sqrt(ρ) * 共通 + np.sqrt(1 - ρ) * 個人)  # 相関ρ
        lo, hi = 信頼区間(標本)
        含んだ += (lo <= 真の母平均 <= hi)
    return 含んだ / 試行

cov_独立 = カバレッジ(ρ=0.0)
cov_相関 = カバレッジ(ρ=0.3)
print(f"独立な標本（ρ=0）　 の95%区間のカバレッジ ： {cov_独立:.1%}  ← 約95%。正しく機能")
print(f"相関した標本（ρ=0.3）の95%区間のカバレッジ ： {cov_相関:.1%}  ← 95%を大きく割る！")

plt.figure(figsize=(6, 4))
plt.bar(["独立 ρ=0", "相関 ρ=0.3\n(空気を読む)"], [cov_独立, cov_相関],
        color=["#3949ab", "#e8503a"])
plt.axhline(0.95, ls="--", color="gray", label="本来あるべき 95%")
plt.ylim(0, 1); plt.ylabel("カバレッジ（真値を含んだ割合）")
plt.title("独立が崩れると『95%信頼区間』は95%を守れない")
for i, v in enumerate([cov_独立, cov_相関]):
    plt.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.legend(); plt.show()

**独立なら約95%。だが空気を読む（相関する）と、『95%信頼区間』は4割程度しか真値を含まない**（半分以下！）。

名前は「95%」でも、中身はまるで違う。狭い区間を見て「こんなに精密に推定できた」と喜ぶのは、独立性を確認していなければ **早とちり**だ。

> 💬 **今期の背骨**
> 
> 区間推定も検定も、すべて「標本は独立」という土台の上に建っている。空気を読む集団から取ったデータでは、その土台が抜けている。**狭い区間・小さいp値は、精密さの証ではなく、独立性の崩壊を見落とした結果かもしれない。** これを疑えることが、統計的誤用を見抜く力だ。


---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| 信頼区間 | $\bar{x}\pm t_{0.975,n-1}\,s/\sqrt{n}$。母分散未知なので t 分布 |
| 95%の意味 | 真値の確率ではなく、手続きを繰り返したときの**カバレッジ**（当たり率） |
| 幅を決めるもの | 標準誤差・信頼度・$n$（独立が前提） |
| 独立が崩れると | 区間が狭すぎてカバレッジが95%を大きく割る＝「95%」が嘘になる |

- 信頼区間は標準誤差から作る。母分散が未知なので t 分布を使う。
- **独立でない標本では、95%信頼区間が95%を守れない**。狭い区間に飛びつかない。

> **課題（Moodle）**：信頼区間の計算・解釈（自動採点）＋「独立でない標本で区間推定するとなぜ危険か」の記述。詳しくはMoodleの第7回課題を見ること。

> **次回予告**：第8回「検定の多重性問題」。「20種類のゼリービーンズを検定したら、緑だけ有意になった」 ―― たくさん試すと、偶然の有意がどんどん出る。p-hackingの罠へ。